# 可转债多因子选券 — 补充实验

本 notebook 包含三类补充实验：手续费敏感性、调仓频率对比、参数网格扫描与鲁棒性分析。
不修改主研究底稿，所有逻辑在本文件内独立运行。

**实验清单**：
1. 手续费敏感性 — 同一策略在不同 fee_rate 下的表现衰减
2. 调仓频率对比 — D / W / BW / M 四种频率的收益-换手权衡
3. 参数网格扫描 — TopN × Fee × 流动性过滤 × 调仓频率 共 180 组组合
4. 鲁棒性排序 — 综合收益与费用敏感度筛选稳健参数


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")

# ---- 中文字体 ----
_fonts = ["Microsoft YaHei", "SimHei", "SimSun", "WenQuanYi Micro Hei", "Noto Sans CJK SC"]
for _f in _fonts:
    try:
        mpl.font_manager.findfont(_f, fallback_to_default=False)
        plt.rcParams["font.sans-serif"] = [_f, "DejaVu Sans"]
        plt.rcParams["axes.unicode_minus"] = False
        break
    except Exception:
        continue

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

DATA_PATH = "../data/cb_data.pq"


In [ ]:
# ============================================================
# 工具函数（统一版本）
# ============================================================

def make_panel(data, value_col):
    """透视表：trade_date × code"""
    return data.pivot(index="trade_date", columns="code", values=value_col).sort_index()


def make_forward_return(price_panel, periods=1):
    """未来收益率矩阵"""
    return price_panel.pct_change(periods=periods, fill_method=None).shift(-periods)


def winsorize_row(row, q=0.02):
    valid = row.dropna()
    if valid.empty:
        return row
    return row.clip(valid.quantile(q), valid.quantile(1 - q))


def zscore_row(row):
    std = row.std()
    if pd.isna(std) or std == 0:
        return row * np.nan
    return (row - row.mean()) / std


def normalize_cross_section(panel, q=0.02):
    return panel.apply(lambda r: zscore_row(winsorize_row(r, q)), axis=1)


def build_composite_score(data, factor_spec, q=0.02):
    scores = []
    for col, sign in factor_spec.items():
        scores.append(normalize_cross_section(make_panel(data, col), q=q) * sign)
    return sum(scores) / len(scores)


def apply_quantile_filter(panel, quantile=0.20, keep="ge"):
    def _row(r):
        out = pd.Series(False, index=r.index)
        valid = r.dropna()
        if valid.empty:
            return out
        th = valid.quantile(quantile)
        out.loc[valid.index] = valid.ge(th) if keep == "ge" else valid.gt(th)
        return out
    return panel.apply(_row, axis=1)


def prepare_research_sample(data):
    mask = pd.Series(True, index=data.index)
    if "amount" in data.columns:
        mask &= data["amount"].fillna(0).gt(0)
    if "turnover" in data.columns:
        mask &= data["turnover"].fillna(0).gt(0)
    if "left_conv_start_days" in data.columns:
        mask &= data["left_conv_start_days"].fillna(np.inf).le(0)
    if "left_years" in data.columns:
        mask &= data["left_years"].fillna(-np.inf).gt(0.3)
    filtered = data.loc[mask].copy()
    print(
        f"研究样本过滤：{len(data):,} → {len(filtered):,} 行 "
        f"({data['trade_date'].nunique()} → {filtered['trade_date'].nunique()} 天)"
    )
    return filtered


def get_rebalance_flag(index, freq="W"):
    """统一的调仓标记：D / W / BW / M"""
    s = pd.Series(index, index=index)
    if freq == "D":
        return pd.Series(True, index=index)
    if freq == "W":
        return s.dt.to_period("W").ne(s.shift(1).dt.to_period("W"))
    if freq == "BW":
        wp = s.dt.to_period("W")
        ids = pd.Series(pd.factorize(wp)[0], index=index)
        return ids.mod(2).eq(0) & wp.ne(wp.shift(1))
    if freq == "M":
        return s.dt.to_period("M").ne(s.shift(1).dt.to_period("M"))
    raise ValueError(f"freq 不支持 '{freq}'，可选 D/W/BW/M")


def topn_buffer_backtest(
    score_panel,
    fwd_ret_panel,
    buy_n=10,
    sell_n=15,
    rebalance_freq="W",
    fee_rate=0.002,
    trade_threshold=None,
):
    """
    统一的 TopN 缓冲回测。

    trade_threshold=None  → 调仓日全量换仓（等权）
    trade_threshold=float → 仅权重变化超过阈值时交易
    """
    rank_panel = score_panel.rank(axis=1, ascending=False, method="first")
    reb = get_rebalance_flag(score_panel.index, freq=rebalance_freq)

    weight = pd.DataFrame(0.0, index=score_panel.index, columns=score_panel.columns)
    prev_w = pd.Series(0.0, index=score_panel.columns)
    holdings = set()

    for dt in score_panel.index:
        if reb.loc[dt]:
            ranks = rank_panel.loc[dt].dropna().sort_values()
            rmap = ranks.to_dict()
            keep = {c for c in holdings if c in rmap and rmap[c] <= sell_n}
            final = [c for c in ranks.index if c in keep or rmap[c] <= buy_n]

            new_w = pd.Series(0.0, index=score_panel.columns)
            if final:
                new_w.loc[final] = 1.0 / len(final)

            if trade_threshold is not None:
                delta = (new_w - prev_w).abs()
                trade = delta > trade_threshold
                updated = prev_w.copy()
                updated.loc[trade[trade].index] = new_w.loc[trade[trade].index]
                total = updated.sum()
                if total > 0:
                    updated /= total
                holdings = set(updated[updated > 0].index)
            else:
                updated = new_w
                holdings = set(final)

            prev_w = updated
            weight.loc[dt] = updated
        else:
            weight.loc[dt] = prev_w

    gross = (weight * fwd_ret_panel).sum(axis=1)
    to = weight.fillna(0).diff().abs().sum(axis=1)
    to.iloc[0] = weight.iloc[0].abs().sum()
    net = gross - fee_rate * to
    nav = (1 + net.fillna(0)).cumprod()
    return pd.DataFrame({"gross_ret": gross, "turnover": to, "net_ret": net, "nav": nav})


def summarize_performance(ret, nav, turnover=None, periods_per_year=244):
    ret = ret.fillna(0)
    nav = nav.ffill().fillna(1.0)
    ann_ret = nav.iloc[-1] ** (periods_per_year / len(nav)) - 1
    ann_vol = ret.std() * np.sqrt(periods_per_year)
    dd = nav / nav.cummax() - 1
    return pd.Series({
        "annual_return": ann_ret,
        "annual_vol": ann_vol,
        "sharpe": ann_ret / ann_vol if ann_vol > 0 else np.nan,
        "max_drawdown": dd.min(),
        "avg_turnover": turnover.mean() if turnover is not None else np.nan,
    })


def smooth_score_ema(panel, span=5):
    """跨时间 EMA 平滑因子得分"""
    return panel.T.ewm(span=span).mean().T


print("工具函数加载完成。")


In [ ]:
# ============================================================
# 数据加载与因子准备
# ============================================================

df = pd.read_parquet(DATA_PATH).reset_index()
for col in ["trade_date", "list_date", "conv_start_date", "conv_end_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
df = df.sort_values(["trade_date", "code"]).reset_index(drop=True)

research_df = prepare_research_sample(df)
price_panel = make_panel(research_df, "close")
fwd_ret_1d = make_forward_return(price_panel, periods=1)

FACTOR_SPEC = {"bond_prem": -1, "dblow": -1, "alpha_pct_chg_5": -1}
composite = build_composite_score(research_df, FACTOR_SPEC, q=0.02)

# 多个流动性过滤版本
amount_panel = make_panel(research_df, "amount")
score_filters = {}
for pct in [20, 30, 40, 50, 60]:
    mask = apply_quantile_filter(amount_panel, quantile=pct / 100, keep="ge")
    key = f"q{pct}"
    score_filters[key] = composite.where(mask)
    n_avg = mask.sum(axis=1).mean()
    print(f"  amount {key}：日均候选 {n_avg:.0f} 只")

print(f"\n研究样本日期范围：{research_df['trade_date'].min().date()} ~ {research_df['trade_date'].max().date()}")


In [ ]:
# ============================================================
# 实验一：手续费敏感性
# ============================================================

fee_list = [0.0, 0.001, 0.002, 0.003]
fee_results = {}
for fee in fee_list:
    fee_results[fee] = topn_buffer_backtest(
        score_filters["q20"],
        fwd_ret_1d,
        buy_n=10,
        sell_n=15,
        rebalance_freq="W",
        fee_rate=fee,
        trade_threshold=0.02,
    )

fee_perf = pd.DataFrame({
    f"fee={fee:.3f}": summarize_performance(
        fee_results[fee]["net_ret"],
        fee_results[fee]["nav"],
        fee_results[fee]["turnover"],
    )
    for fee in fee_list
}).T
fee_perf.index.name = " "

print("手续费敏感性 — Top10/Top15, amount q20, 周调仓, trade_threshold=0.02")
display(
    fee_perf.style.format({
        "annual_return": "{:.2%}",
        "annual_vol": "{:.2%}",
        "sharpe": "{:.4f}",
        "max_drawdown": "{:.2%}",
        "avg_turnover": "{:.4f}",
    })
)

# ---- 图表 ----
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
colors = ["#2c7bb6", "#abd9e9", "#fdae61", "#d7191c"]

for (fee, res), c in zip(fee_results.items(), colors):
    axes[0].plot(res["nav"].values, color=c, label=f"fee={fee:.3f}", linewidth=1.2)
axes[0].set_title("不同手续费下的策略净值", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

for j, (m, lbl) in enumerate(zip(
    ["annual_return", "sharpe"], ["年化收益", "夏普比率"]
)):
    vals = [fee_perf.loc[f"fee={f:.3f}", m] for f in fee_list]
    axes[1].plot(
        fee_list, vals, "o-",
        color=["#2c7bb6", "#d7191c"][j],
        markersize=8, linewidth=2, label=lbl,
    )
    for x, y in zip(fee_list, vals):
        axes[1].annotate(
            f"{y:.1%}" if "return" in m else f"{y:.2f}",
            (x, y), textcoords="offset points", xytext=(0, 12),
            fontsize=8, ha="center",
        )
axes[1].set_xlabel("手续费率")
axes[1].set_title("收益与夏普随手续费衰减", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(1.0))

fig.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 实验二：调仓频率对比
# 正式口径：research_df, amount q20, Top10/Top15, fee=0.002
# ============================================================

freq_list = ["D", "W", "BW", "M"]
freq_results = {}
for freq in freq_list:
    freq_results[freq] = topn_buffer_backtest(
        score_filters["q20"],
        fwd_ret_1d,
        buy_n=10,
        sell_n=15,
        rebalance_freq=freq,
        fee_rate=0.002,
        trade_threshold=None,
    )

freq_perf = pd.DataFrame([
    summarize_performance(
        freq_results[f]["net_ret"],
        freq_results[f]["nav"],
        freq_results[f]["turnover"],
    )
    for f in freq_list
], index=freq_list)
freq_perf.index.name = "freq"

print("调仓频率对比 — 正式口径：research_df, q20, Top10/Top15, fee=0.002")
display(
    freq_perf.style.format({
        "annual_return": "{:.2%}",
        "annual_vol": "{:.2%}",
        "sharpe": "{:.4f}",
        "max_drawdown": "{:.2%}",
        "avg_turnover": "{:.4f}",
    })
)

# ---- 图表 ----
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fc = {"D": "#d7191c", "W": "#fdae61", "BW": "#2c7bb6", "M": "#5e3c99"}

for freq, res in freq_results.items():
    axes[0].plot(res["nav"].values, color=fc[freq], label=freq, linewidth=1.2)
axes[0].set_title("不同调仓频率的净值曲线 (fee=0.002)", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

x = np.arange(len(freq_list))
w = 0.35
axes[1].bar(x - w / 2, freq_perf["annual_return"], w, color="#2c7bb6",
            alpha=0.85, label="年化收益")
axes[1].bar(x + w / 2, freq_perf["avg_turnover"], w, color="#fdae61",
            alpha=0.85, label="平均换手")
for i, (v1, v2) in enumerate(
    zip(freq_perf["annual_return"], freq_perf["avg_turnover"])
):
    axes[1].text(i - w / 2, v1 + 0.01, f"{v1:.1%}", ha="center", fontsize=8)
    axes[1].text(i + w / 2, v2 + 0.01, f"{v2:.2f}", ha="center", fontsize=8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(freq_list)
axes[1].set_title("年化收益 vs 平均换手", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis="y")

fig.tight_layout()
plt.show()

print("结论：BW 双周在收益与换手之间最均衡。")


In [ ]:
# ============================================================
# 实验三：参数网格扫描
# TopN(3) × Fee(3) × Filter(5) × Freq(4) = 180 组
# ============================================================

topn_list = [10, 15, 20]
fee_list = [0.001, 0.002, 0.003]
freq_list = ["D", "W", "BW", "M"]
filter_keys = ["q20", "q30", "q40", "q50", "q60"]

grid_rows = []
n_total = len(topn_list) * len(fee_list) * len(filter_keys) * len(freq_list)

for topn in topn_list:
    for fee in fee_list:
        for fk in filter_keys:
            for freq in freq_list:
                res = topn_buffer_backtest(
                    score_filters[fk],
                    fwd_ret_1d,
                    buy_n=topn,
                    sell_n=min(topn + 5, int(topn * 1.5)),
                    rebalance_freq=freq,
                    fee_rate=fee,
                    trade_threshold=0.02,
                )
                perf = summarize_performance(
                    res["net_ret"], res["nav"], res["turnover"]
                )
                grid_rows.append({
                    "topn": topn, "fee": fee, "filter": fk, "freq": freq,
                    **perf.to_dict(),
                })

grid_df = pd.DataFrame(grid_rows)
print(f"参数网格扫描完成：{len(grid_df)} / {n_total} 组")

print("\n年化收益前 10 组：")
display(
    grid_df.nlargest(10, "annual_return")[
        ["topn", "fee", "filter", "freq",
         "annual_return", "sharpe", "max_drawdown", "avg_turnover"]
    ].style.format({
        "annual_return": "{:.2%}",
        "sharpe": "{:.4f}",
        "max_drawdown": "{:.2%}",
        "avg_turnover": "{:.4f}",
    })
)


In [ ]:
# ============================================================
# 网格可视化
# ============================================================

fig = plt.figure(figsize=(16, 7))

# 左图：全网格散点
ax1 = fig.add_subplot(1, 2, 1)
markers = {10: "o", 15: "s", 20: "^"}
for topn, mk in markers.items():
    sub = grid_df[grid_df["topn"] == topn]
    ax1.scatter(
        sub["max_drawdown"].abs(), sub["annual_return"],
        c=sub["fee"], cmap="YlOrRd", s=70, marker=mk,
        alpha=0.7, edgecolors="white", linewidth=0.3,
        label=f"Top{topn}",
    )
ax1.set_xlabel("最大回撤（绝对值）", fontsize=11)
ax1.set_ylabel("年化收益", fontsize=11)
ax1.set_title("全网格：收益 vs 回撤\n颜色=手续费率  |  标记=持仓数", fontsize=12, fontweight="bold")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# 右图：热力图 — fee=0.002, BW
ax2 = fig.add_subplot(1, 2, 2)
heat = grid_df[(grid_df["fee"] == 0.002) & (grid_df["freq"] == "BW")]
ht = heat.pivot_table(values="annual_return", index="topn", columns="filter")
im = ax2.imshow(ht.values, cmap="RdYlGn", aspect="auto", vmin=0.05, vmax=0.35)
ax2.set_xticks(range(len(ht.columns)))
ax2.set_xticklabels(ht.columns, fontsize=10)
ax2.set_yticks(range(len(ht.index)))
ax2.set_yticklabels([f"Top{n}" for n in ht.index], fontsize=10)
ax2.set_title("fee=0.002, BW：TopN × 流动性过滤 收益热力图", fontsize=12, fontweight="bold")
for i in range(len(ht.index)):
    for j in range(len(ht.columns)):
        ax2.text(j, i, f"{ht.iloc[i, j]:.1%}",
                 ha="center", va="center", fontsize=9, fontweight="bold")
plt.colorbar(im, ax=ax2).set_label("年化收益", fontsize=9)

fig.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 实验四：鲁棒性分析
# 费率上升时各参数组合的收益保持能力
# ============================================================

robust_rows = []
for topn in [10, 15, 20]:
    for fk in ["q20", "q30"]:
        for freq in ["W", "BW", "M"]:
            sub = grid_df[
                (grid_df["topn"] == topn)
                & (grid_df["filter"] == fk)
                & (grid_df["freq"] == freq)
            ].sort_values("fee")
            if len(sub) < 3:
                continue
            r1 = sub[sub["fee"] == 0.001]["annual_return"].values[0]
            r2 = sub[sub["fee"] == 0.002]["annual_return"].values[0]
            r3 = sub[sub["fee"] == 0.003]["annual_return"].values[0]
            slope = (abs(r2 - r1) + abs(r3 - r2)) / 2 / 0.001
            retention = r3 / r2 if r2 > 0 else 0
            robust_rows.append({
                "topn": topn, "filter": fk, "freq": freq,
                "ret_001": r1, "ret_002": r2, "ret_003": r3,
                "slope": slope, "retention": retention,
                "score": r2 * retention,
            })

robust_df = pd.DataFrame(robust_rows).sort_values("score", ascending=False)
print("鲁棒性综合评分（fee=0.002 收益 × 费率上升保持率）\n")
display(
    robust_df.head(10).style.format({
        "ret_001": "{:.2%}", "ret_002": "{:.2%}", "ret_003": "{:.2%}",
        "slope": "{:.2f}", "retention": "{:.2%}", "score": "{:.4f}",
    })
)

# 衰减曲线
fig, ax = plt.subplots(figsize=(10, 5))
for i in range(min(3, len(robust_df))):
    r = robust_df.iloc[i]
    label = f"Top{int(r['topn'])}, {r['filter']}, {r['freq']}"
    ax.plot([0.001, 0.002, 0.003], [r["ret_001"], r["ret_002"], r["ret_003"]],
            "o-", linewidth=2, markersize=8, label=label)
ax.set_xlabel("手续费率", fontsize=11)
ax.set_ylabel("年化收益", fontsize=11)
ax.set_title("Top 3 鲁棒参数：费率上升时的收益衰减", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
fig.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 小结
# ============================================================

best = robust_df.iloc[0]
print(f"""
═══ 补充实验结论 ═══

1. 手续费敏感性：
   手续费每提高 0.1%，年化收益约下降 5–7 个百分点。
   fee=0.002 时策略仍能保持显著正超额。

2. 最优调仓频率：
   BW 双周以收益可观 + 换手可控综合最优。
   日频换手过高（~0.76），月频信号衰减明显。

3. 鲁棒性最优参数：
   Top{int(best['topn'])}, amount {best['filter']}, {best['freq']}调仓
   → fee=0.002 下收益 {best['ret_002']:.2%}，
     fee=0.003 下仍保持 {best['ret_003']:.2%}

4. 与主策略的关系：
   本 notebook 的鲁棒性分析支持了主策略选用 BW + Top10/Top15 的合理性。
   trade_threshold=0.02 有一定减换手效果但差异不大，故未进入主策略。
""")


---
## 附录：参数定义与口径

| 项目 | 取值 |
|------|------|
| 因子 | `bond_prem + dblow + alpha_pct_chg_5` |
| 因子方向 | 统一负向（分数越高越好） |
| 合成方式 | 日截面去极值 (2%) → z-score → 等权平均 |
| 回测函数 | 统一的 `topn_buffer_backtest`，支持有 / 无 `trade_threshold` |
| 数据范围 | 2018-01-02 ~ 2024-12-26，`research_df` 全量过滤 |
| 环境 | `QuantEnv`，详见项目 `requirements.txt` |
